# Tensor Operations

| Difficulty | ████░░░░░░ 4/10 |
| :--- | :--- |
| Prerequisites | 01.02 |
| Time | ~70 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/01_Linear_Algebra/09_tensor_operations.ipynb)

---

## 🎯 Learning Objective

Extend vectors and matrices to tensors, and learn Einstein summation (einsum) notation as the universal language for the batched, multi-dimensional operations inside neural networks.

---

## 📐 Theory

A **tensor** generalizes scalars (rank 0), vectors (rank 1), and matrices (rank 2) to
arbitrarily many indices. A rank-3 tensor $T_{ijk}$ is naturally visualized as a stack of
matrices; a batch of RGB images is a rank-4 tensor indexed by (batch, channel, height, width).
Each index position is called a **mode** or **axis**.

### Slicing and contraction

Fixing some indices and letting others vary is **slicing**: $T_{i,:,:}$ extracts the $i$-th
matrix "sheet." **Contraction** sums over a shared index between two tensors — matrix
multiplication is the simplest example, contracting the shared index $j$ in $C_{ik} = \sum_j
A_{ij}B_{jk}$.

### Einstein summation (einsum) notation

Einsum notation makes contractions explicit and compact by writing only the indices, with any
index that's *repeated but not in the output* implicitly summed over:

| Einsum string | Operation |
|---|---|
| `'ij,jk->ik'` | matrix multiplication |
| `'ij->ji'` | transpose |
| `'ii->'` | trace (sum of diagonal) |
| `'i,i->'` | dot product |
| `'i,j->ij'` | outer product |
| `'bij,bjk->bik'` | **batched** matrix multiplication (the `b` index is kept, not summed) |

This single notation subsumes dot products, matrix multiplication, batched operations, and
outer products — which is exactly why `torch.einsum`/`np.einsum` shows up everywhere in
real model code instead of chains of `reshape`/`transpose`/`matmul`.

### Outer product

The outer product $\mathbf{u}\otimes\mathbf{v}$ of an $m$-vector and $n$-vector produces an
$m\times n$ **rank-1** matrix: $(\mathbf{u}\otimes\mathbf{v})_{ij} = u_i v_j$. Sums of a few
outer products are exactly how we built low-rank matrices in
[Notebook 01.02](02_matrix_operations.ipynb) and [01.07](07_SVD_and_PCA.ipynb).

---

## 👁️ Visual Intuition

A rank-3 tensor is a stack of matrices — slicing along the first axis peels off one matrix
"sheet" at a time.

In [ ]:
# Setup + visualizing a rank-3 tensor as a stack of 2D slices
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

rng = np.random.default_rng(0)
T = rng.standard_normal((4, 5, 5))   # a rank-3 tensor: 4 "sheets", each a 5x5 matrix

fig, axes = plt.subplots(1, 4, figsize=(13, 3.5))
for i, ax in enumerate(axes):
    im = ax.imshow(T[i], cmap="coolwarm", vmin=T.min(), vmax=T.max())
    ax.set_title(f"T[{i}, :, :]  (slice {i})")
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle(f"Tensor T has shape {T.shape} -- a stack of {T.shape[0]} matrices, each {T.shape[1]}x{T.shape[2]}")
plt.tight_layout()
plt.show()

## 🐍 Implementation from Scratch

We implement matrix multiplication, batched matrix multiplication, and outer products first
as explicit loops (making the summed/free indices completely explicit), then show the
one-line `einsum` equivalent for each.

In [ ]:
# Explicit-loop tensor contractions vs. their einsum equivalents
import numpy as np

rng = np.random.default_rng(0)

# --- Plain matrix multiply: C_ik = sum_j A_ij B_jk ---
A = rng.standard_normal((3, 4))
B = rng.standard_normal((4, 5))
C_loop = np.zeros((3, 5))
for i in range(3):
    for k in range(5):
        C_loop[i, k] = sum(A[i, j] * B[j, k] for j in range(4))
C_einsum = np.einsum('ij,jk->ik', A, B)
print("Matmul loop matches einsum('ij,jk->ik'):", np.allclose(C_loop, C_einsum))
print("Both match A @ B:                       ", np.allclose(C_loop, A @ B))

# --- Batched matrix multiply: C_bik = sum_j A_bij B_bjk  (batch index b is NOT summed) ---
batch = 2
Ab = rng.standard_normal((batch, 3, 4))
Bb = rng.standard_normal((batch, 4, 5))
Cb_loop = np.zeros((batch, 3, 5))
for b in range(batch):
    for i in range(3):
        for k in range(5):
            Cb_loop[b, i, k] = sum(Ab[b, i, j] * Bb[b, j, k] for j in range(4))
Cb_einsum = np.einsum('bij,bjk->bik', Ab, Bb)
print("\nBatched matmul loop matches einsum('bij,bjk->bik'):", np.allclose(Cb_loop, Cb_einsum))

# --- Outer product: (u ox v)_ij = u_i v_j ---
u, v = rng.standard_normal(3), rng.standard_normal(4)
outer_loop = np.array([[u[i] * v[j] for j in range(4)] for i in range(3)])
outer_einsum = np.einsum('i,j->ij', u, v)
print("\nOuter product loop matches einsum('i,j->ij'):", np.allclose(outer_loop, outer_einsum))
print("Both match np.outer:                          ", np.allclose(outer_loop, np.outer(u, v)))

## 🤖 Why This Matters for AI

**Multi-head attention** splits queries/keys/values into `n_heads` independent chunks, computes
attention scores per head in parallel, then concatenates the results. This is naturally a
rank-4 tensor operation: shape `(batch, heads, seq_len, head_dim)`. The einsum string
`'bhid,bhjd->bhij'` computes every head's attention scores for every batch element in one call
— no manual loop over batches or heads required.

In practice, `torch.einsum('bhid,bhjd->bhij', Q, K)` computes attention scores across all heads in one line — this is the pattern used in HuggingFace's `transformers` library and Karpathy's nanoGPT, mapping directly to the index conventions from this notebook's Theory section.

---


In [ ]:
# Multi-head attention SCORES computed with a single einsum call over (batch, heads, seq, dim)
import numpy as np

rng = np.random.default_rng(0)
batch, n_heads, seq_len, head_dim = 2, 4, 6, 8

Q = rng.standard_normal((batch, n_heads, seq_len, head_dim))
K = rng.standard_normal((batch, n_heads, seq_len, head_dim))

# scores[b,h,i,j] = sum_d Q[b,h,i,d] * K[b,h,j,d]  -- every (query i, key j) pair, per batch & head
scores_einsum = np.einsum('bhid,bhjd->bhij', Q, K) / np.sqrt(head_dim)

# Explicit loop version, to make the contraction completely unambiguous
scores_loop = np.zeros((batch, n_heads, seq_len, seq_len))
for b in range(batch):
    for h in range(n_heads):
        scores_loop[b, h] = (Q[b, h] @ K[b, h].T) / np.sqrt(head_dim)

print("Q shape:", Q.shape, " K shape:", K.shape)
print("Attention scores shape:", scores_einsum.shape, "  (batch, heads, query_pos, key_pos)")
print("einsum matches explicit per-batch-per-head loop:", np.allclose(scores_einsum, scores_loop))
print("\nOne einsum call replaced", batch * n_heads, "separate matrix multiplications.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. Given vectors $u=(1,2,3)$ and $v=(4,5)$, compute their outer product by hand as a
   $3\times2$ matrix, then verify with `np.einsum('i,j->ij', u, v)`. As a shape check, predict
   the result's shape from $u$ and $v$'s lengths alone before computing anything.

<details>
<summary>💡 Solution</summary>

Shape prediction: $u$ has length 3, $v$ has length 2, so the outer product must have shape
$(3,2)$ — one row per entry of $u$, one column per entry of $v$. By hand,
$(\mathbf{u}\otimes\mathbf{v})_{ij}=u_iv_j$:
$$\begin{pmatrix}1\cdot4&1\cdot5\\2\cdot4&2\cdot5\\3\cdot4&3\cdot5\end{pmatrix} =
\begin{pmatrix}4&5\\8&10\\12&15\end{pmatrix},$$
shape $(3,2)$ as predicted, matching `np.einsum('i,j->ij', u, v)` and `np.outer(u, v)` exactly.

</details>

### 💭 UNDERSTAND
2. The notebook's multi-head attention example computes `scores_einsum =
   np.einsum('bhid,bhjd->bhij', Q, K) / sqrt(head_dim)`. Without running anything yet: if you
   instead wrote the output indices in a different order, `'bhid,bhjd->bhji'`, would you get the
   *same* array back, or a *related-but-different* one? What specific operation connects the two
   results? Then check whether `scores[b,h]` (the $i,j$ matrix for one batch/head) is itself
   symmetric when $Q$ and $K$ are different tensors, versus when $Q$ and $K$ are the *same*
   tensor.

<details>
<summary>💡 Solution</summary>

`'bhid,bhjd->bhji'` is *not* the same array as `'bhid,bhjd->bhij'` — the two differ by swapping
which of $i,j$ indexes rows versus columns, so `scores_bhji` is exactly the transpose of
`scores_bhij` on its last two axes (verified: `np.swapaxes(scores_bhji, -1, -2)` matches
`scores_bhij` exactly). Reordering output indices in einsum never changes *which numbers* get
computed, only how they're laid out in the result array. As for symmetry: with $Q\ne K$ (the
general case), `scores[b,h]` is *not* symmetric (query $i$ attending to key $j$ has no reason to
equal query $j$ attending to key $i$, since they involve different vectors from $Q$ and $K$
respectively) — but with $Q=K$ (i.e. computing `einsum('bhid,bhjd->bhij', Q, Q)`, closer to a
self-similarity matrix), the result *is* symmetric, because $Q_i\cdot Q_j=Q_j\cdot Q_i$ for any
two vectors from the *same* tensor — ordinary dot-product commutativity, not anything special
about einsum.

</details>

### ✏️ DERIVE
3. Derive a general formula for the number of scalar multiplications a two-tensor einsum
   contraction performs, in terms of the sizes of its free (kept) and summed (contracted)
   indices. Apply it to `'bhid,bhjd->bhij'` with `batch=2, n_heads=4, seq_len=6, head_dim=8`.

<details>
<summary>💡 Solution outline</summary>

Every entry of the output array requires one multiplication per value of each *summed* index
(here, only $d$, ranging over `head_dim`), and the output array itself has one entry per
combination of *free* (kept) indices (here, $b,h,i,j$). So the total multiplication count is
$$(\text{product of free-index sizes}) \times (\text{product of summed-index sizes}).$$
For `'bhid,bhjd->bhij'`: free indices are $b,h,i,j$ (sizes $2,4,6,6$), summed index is $d$ (size
$8$), giving $2\times4\times6\times6\times8=2304$ multiplications. Verified directly by counting
multiplications in an explicit 5-nested-loop version of the same contraction: exactly $2304$,
matching the formula precisely — confirming the formula isn't just a rough estimate. This is
the same reasoning that makes multi-head attention's cost scale as
$O(\text{batch}\times\text{heads}\times\text{seq\_len}^2\times\text{head\_dim})$: it's a direct
reading of the free ($b,h,i,j$) versus summed ($d$) indices in exactly this einsum string.

</details>

### 🐍 IMPLEMENT
4. Extend the multi-head attention example: after computing `scores_einsum`, apply softmax
   along the last axis (key positions) to get attention weights, then use a second einsum call,
   `'bhij,bhjd->bhid'`, to combine the weights with a value tensor `V` of the same shape as `Q`
   and `K`, producing the attended output.

<details>
<summary>✅ How to know you're right</summary>

The output's shape must exactly match `Q`'s shape, `(batch, n_heads, seq_len, head_dim)` — the
attention mechanism reweights and mixes value vectors but never changes the tensor's overall
shape. Verify the einsum against an explicit loop over `(b, h)` that does ordinary 2D matrix
multiplication (`weights[b, h] @ V[b, h]`) for each batch/head slice — the two must agree
exactly, since `'bhij,bhjd->bhid'` for fixed $b,h$ is exactly the 2D contraction
`weights @ V`. As an additional sanity check independent of the loop comparison: since softmax
weights sum to 1 along the key axis, the attended output for any query position must be a
genuine weighted average of $V$'s row vectors — so each output vector's norm should never exceed
the largest norm among $V$'s row vectors for that batch/head (a weighted average can't exceed
its largest input in norm, by the triangle inequality).

</details>

### 🤖 APPLY
5. An empirical Fisher information matrix approximation for a $d$-dimensional parameter is
   built by averaging the outer product of many per-example gradient vectors:
   $\hat{F}=\frac1N\sum_{n}\mathbf{g}_n\mathbf{g}_n^\top$. Given a `(N, d)` array `G` of `N`
   per-example gradients, compute $\hat F$ using `np.einsum('bi,bj->ij', G, G) / N` and confirm
   it matches both a direct `(G.T @ G) / N` computation and the mean of `N` individually-formed
   `(d, d)` outer products (via `np.einsum('bi,bj->bij', G, G)`, kept per-example, then
   averaged).

<details>
<summary>✅ What you should observe</summary>

All three routes — the direct einsum contraction, `(G.T @ G) / N`, and the mean of `N` separate
per-example outer products — should agree exactly (to floating-point precision), since they're
three ways of computing the identical sum. As a structural sanity check on $\hat F$ itself
(not just that the three computations agree with each other): $\hat F$ must come out symmetric
(any sum of outer products $\mathbf{g}\mathbf{g}^\top$ is automatically symmetric, since
$(\mathbf{g}\mathbf{g}^\top)^\top=\mathbf{g}\mathbf{g}^\top$) and positive-semi-definite (its
eigenvalues should all be $\ge0$ — verified directly via `np.linalg.eigvalsh`) — both properties
that a genuine (empirical) Fisher information / covariance-like matrix must have by
construction, regardless of what the underlying gradients actually look like.

</details>

### 🚀 CHALLENGE
6. Contracting a chain of three tensors, $A(n\times r)$, $B(r\times n)$, $C(n\times r)$ with
   $r\ll n$, can be done as `(A @ B) @ C` or `A @ (B @ C)` — mathematically identical by
   associativity, but not computationally identical. Time both approaches for a large $n$ (e.g.
   $n=4000$, $r=3$), and separately time `np.einsum('ij,jk,kl->il', A, B, C, optimize=False)`
   against the same call with `optimize=True`. Explain the size of whatever difference you find
   in terms of the shapes of the *intermediate* results each approach has to form.

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer measures a dramatic difference — `(A@B)@C` (which first forms the huge
$n\times n$ intermediate `A@B`) can take orders of magnitude longer than `A@(B@C)` (which first
forms the tiny $r\times r$ intermediate `B@C`) for $n\gg r$, and `einsum(..., optimize=False)`
similarly lags far behind `optimize=True` on the same contraction, converging to a comparably
large speedup. A strong answer explains this isn't just "more multiplications" in a vague sense
but ties it to the actual intermediate shapes: `(A@B)@C` forms an $n\times n$ intermediate
(here, $4000\times4000$ — 16 million entries) purely as scratch work, discarded right after use,
while `A@(B@C)` only ever forms an $r\times r$ intermediate (here, $3\times3$ — 9 entries) —
and a rough FLOP count for the dominant matrix-multiply terms gives a factor-of-2 difference
($2n^2r$ vs. $n^2r+nr^2\approx n^2r$), which under-explains the full size of the actual
wall-clock gap, because forming and reading back a huge intermediate array also costs memory
bandwidth and allocation time that a pure FLOP count doesn't capture. The general lesson —
`optimize=True`'s actual job — is that when contracting a *chain* of tensors, the order in which
you contract them can change the total work by orders of magnitude depending on the shapes
involved, exactly the same issue library implementations of chained matrix products (and framework
compute-graph optimizers, more generally) actively search for a good contraction order to avoid.

</details>

---

## 📚 Further Reading
- [einsum is all you need (blog post with visual explanations)](https://rockt.github.io/2018/04/30/einsum)
- Vaswani et al., ["Attention Is All You Need"](https://arxiv.org/abs/1706.03762)

---

*Next notebook: [Linear Algebra in Transformers](10_linear_algebra_in_transformers.ipynb)*